In [8]:
from pathlib import Path
import os
import math
import sqlite3
import datetime as dt
import numpy as np
import pandas as pd

BASE_DIR = Path.cwd()
SDM_DB = BASE_DIR / "SDM.db"
TARGET_DB = BASE_DIR / "DWS.db"
LOAD_TS = dt.datetime.now().replace(microsecond=0).isoformat()

print("SDM:", SDM_DB)
print("Doel-DWH:", TARGET_DB)
print("Load timestamp:", LOAD_TS)

SDM: C:\Users\Bayra\Documents\GitHub\DEAI-Duo\DWH week 5\SDM.db
Doel-DWH: C:\Users\Bayra\Documents\GitHub\DEAI-Duo\DWH week 5\DWS.db
Load timestamp: 2026-03-26T17:52:32


In [9]:
# Helperfuncties
def connect(db):
    con = sqlite3.connect(db)
    con.execute("PRAGMA foreign_keys = ON;")
    return con

def read_table(con, table):
    return pd.read_sql_query(f"SELECT * FROM {table}", con)

def age_from_birthdate(date_str, ref_date=dt.date(2024, 12, 31)):
    if pd.isna(date_str) or date_str is None:
        return None
    d = dt.date.fromisoformat(str(date_str)[:10])
    return ref_date.year - d.year - ((ref_date.month, ref_date.day) < (d.month, d.day))

def age_cat(age):
    if age is None or pd.isna(age):
        return None
    age = int(age)
    if age < 18:
        return "<18"
    if age <= 30:
        return "18-30"
    if age <= 50:
        return "31-50"
    return "51+"

def season(month):
    return {
        12: 'Winter', 1: 'Winter', 2: 'Winter',
        3: 'Lente', 4: 'Lente', 5: 'Lente',
        6: 'Zomer', 7: 'Zomer', 8: 'Zomer',
        9: 'Herfst', 10: 'Herfst', 11: 'Herfst'
    }[int(month)]

def dagdeel(hour):
    hour = int(hour)
    if hour < 12:
        return 'Ochtend'
    if hour < 18:
        return 'Middag'
    return 'Avond'

def normalize_time(t):
    return str(t).split('.')[0]

def minutes_between(start, end):
    fmt = '%H:%M:%S'
    st = dt.datetime.strptime(normalize_time(start), fmt)
    en = dt.datetime.strptime(normalize_time(end), fmt)
    return int((en - st).total_seconds() // 60)

In [10]:
# DWH-schema opbouwen
def prepare_target_schema(db):
    if Path(db).exists():
        os.remove(db)

    con = connect(db)
    con.executescript("""
    CREATE TABLE Dim_Product (
        ProductKey      INTEGER PRIMARY KEY AUTOINCREMENT,
        BronSysteem     TEXT NOT NULL,
        ProductNr       INTEGER NOT NULL,
        ProductType     TEXT NOT NULL CHECK (ProductType IN ('Fiets', 'Accessoire')),
        Naam            TEXT,
        Merk            TEXT,
        Soort           TEXT,
        Type            TEXT,
        Kleur           TEXT,
        Standaardprijs  REAL,
        Inkoopprijs     REAL,
        ValidFrom       TEXT NOT NULL,
        ValidTo         TEXT,
        IsCurrent       INTEGER NOT NULL DEFAULT 1 CHECK (IsCurrent IN (0,1))
    );
    CREATE UNIQUE INDEX UX_Dim_Product_Current
        ON Dim_Product (BronSysteem, ProductType, ProductNr)
        WHERE IsCurrent = 1;

    CREATE TABLE Dim_Partner (
        PartnerKey      INTEGER PRIMARY KEY AUTOINCREMENT,
        BronSysteem     TEXT NOT NULL,
        PartnerNr       INTEGER NOT NULL,
        PartnerType     TEXT NOT NULL CHECK (PartnerType IN ('Fabrikant', 'Leverancier')),
        Naam            TEXT,
        Adres           TEXT,
        Plaats          TEXT,
        UNIQUE (BronSysteem, PartnerType, PartnerNr)
    );

    CREATE TABLE Dim_Klant (
        KlantKey                INTEGER PRIMARY KEY AUTOINCREMENT,
        BronSysteem             TEXT NOT NULL,
        KlantNr                 INTEGER NOT NULL,
        Naam                    TEXT NOT NULL,
        Adres                   TEXT,
        Woonplaats              TEXT,
        Geslacht                TEXT CHECK (Geslacht IN ('M', 'V', 'X') OR Geslacht IS NULL),
        Geboortedatum           TEXT,
        Leeftijd                INTEGER,
        Leeftijdscategorie      TEXT,
        ValidFrom               TEXT NOT NULL,
        ValidTo                 TEXT,
        IsCurrent               INTEGER NOT NULL DEFAULT 1 CHECK (IsCurrent IN (0,1))
    );
    CREATE UNIQUE INDEX UX_Dim_Klant_Current
        ON Dim_Klant (BronSysteem, KlantNr)
        WHERE IsCurrent = 1;

    CREATE TABLE Dim_Monteur (
        MonteurKey      INTEGER PRIMARY KEY AUTOINCREMENT,
        BronSysteem     TEXT NOT NULL,
        MonteurNr       INTEGER NOT NULL,
        Naam            TEXT NOT NULL,
        Woonplaats      TEXT,
        Uurloon         REAL,
        UNIQUE (BronSysteem, MonteurNr)
    );

    CREATE TABLE Dim_Filiaal (
        FiliaalKey      INTEGER PRIMARY KEY AUTOINCREMENT,
        BronSysteem     TEXT NOT NULL,
        FiliaalNr       INTEGER NOT NULL,
        Naam            TEXT NOT NULL,
        Adres           TEXT,
        Provincie       TEXT,
        UNIQUE (BronSysteem, FiliaalNr)
    );

    CREATE TABLE Dim_Datum (
        DatumKey        INTEGER PRIMARY KEY AUTOINCREMENT,
        Datum           TEXT NOT NULL UNIQUE,
        Dag             INTEGER NOT NULL,
        Maand           INTEGER NOT NULL,
        Jaar            INTEGER NOT NULL,
        Kwartaal        INTEGER NOT NULL,
        Weekdag         TEXT NOT NULL,
        Seizoen         TEXT,
        IsWeekend       INTEGER NOT NULL CHECK (IsWeekend IN (0,1))
    );

    CREATE TABLE Dim_Tijd (
        TijdKey         INTEGER PRIMARY KEY AUTOINCREMENT,
        Tijd            TEXT NOT NULL UNIQUE,
        Uur             INTEGER NOT NULL,
        Minuut          INTEGER NOT NULL,
        Dagdeel         TEXT
    );

    CREATE TABLE Fact_Inkoop (
        InkoopKey           INTEGER PRIMARY KEY AUTOINCREMENT,
        BronFeit            TEXT NOT NULL,
        InkoopNr            INTEGER NOT NULL,
        ProductKey          INTEGER NOT NULL,
        PartnerKey          INTEGER NOT NULL,
        DatumKey            INTEGER NOT NULL,
        Aantal              INTEGER NOT NULL,
        Inkoopprijs         REAL NOT NULL,
        Inkoopbedrag        REAL NOT NULL,
        KortingBedrag       REAL,
        FOREIGN KEY (ProductKey) REFERENCES Dim_Product(ProductKey),
        FOREIGN KEY (PartnerKey) REFERENCES Dim_Partner(PartnerKey),
        FOREIGN KEY (DatumKey) REFERENCES Dim_Datum(DatumKey),
        UNIQUE (BronFeit, InkoopNr)
    );

    CREATE TABLE Fact_Verkoop (
        VerkoopKey          INTEGER PRIMARY KEY AUTOINCREMENT,
        BronFeit            TEXT NOT NULL,
        VerkoopNr           INTEGER NOT NULL,
        ProductKey          INTEGER NOT NULL,
        KlantKey            INTEGER NOT NULL,
        MonteurKey          INTEGER NOT NULL,
        FiliaalKey          INTEGER NOT NULL,
        DatumKey            INTEGER NOT NULL,
        Aantal              INTEGER NOT NULL,
        Verkoopprijs        REAL NOT NULL,
        Omzet               REAL NOT NULL,
        Inkoopbedrag        REAL,
        Brutowinst          REAL,
        FOREIGN KEY (ProductKey) REFERENCES Dim_Product(ProductKey),
        FOREIGN KEY (KlantKey) REFERENCES Dim_Klant(KlantKey),
        FOREIGN KEY (MonteurKey) REFERENCES Dim_Monteur(MonteurKey),
        FOREIGN KEY (FiliaalKey) REFERENCES Dim_Filiaal(FiliaalKey),
        FOREIGN KEY (DatumKey) REFERENCES Dim_Datum(DatumKey),
        UNIQUE (BronFeit, VerkoopNr)
    );

    CREATE TABLE Fact_Onderhoud (
        OnderhoudKey            INTEGER PRIMARY KEY AUTOINCREMENT,
        BronFeit                TEXT NOT NULL DEFAULT 'Onderhoud',
        OnderhoudNr             INTEGER NOT NULL UNIQUE,
        ProductKey              INTEGER NOT NULL,
        MonteurKey              INTEGER NOT NULL,
        FiliaalKey              INTEGER NOT NULL,
        DatumKey                INTEGER NOT NULL,
        StartTijdKey            INTEGER NOT NULL,
        EindTijdKey             INTEGER NOT NULL,
        AantalOnderhoud         INTEGER NOT NULL DEFAULT 1,
        OnderhoudsduurMin       INTEGER NOT NULL,
        Arbeidskosten           REAL NOT NULL,
        FOREIGN KEY (ProductKey) REFERENCES Dim_Product(ProductKey),
        FOREIGN KEY (MonteurKey) REFERENCES Dim_Monteur(MonteurKey),
        FOREIGN KEY (FiliaalKey) REFERENCES Dim_Filiaal(FiliaalKey),
        FOREIGN KEY (DatumKey) REFERENCES Dim_Datum(DatumKey),
        FOREIGN KEY (StartTijdKey) REFERENCES Dim_Tijd(TijdKey),
        FOREIGN KEY (EindTijdKey) REFERENCES Dim_Tijd(TijdKey)
    );
    """)
    con.commit()
    con.close()

In [11]:
# Brontabellen extraheren en transformeren naar ETL-views
def extract_sources(sdm_db):
    con = sqlite3.connect(sdm_db)

    fi = read_table(con, 'Fiets_Inkoop_Fiets')
    fi['BronSysteem'] = 'Fiets_Inkoop'; fi['ProductType'] = 'Fiets'; fi['Naam'] = None
    fi = fi.rename(columns={'fietsnr':'ProductNr','type':'Type','standaardprijs':'Standaardprijs','inkoopprijs':'Inkoopprijs','merk':'Merk','soort':'Soort','kleur':'Kleur'})
    fi = fi[['BronSysteem','ProductNr','ProductType','Naam','Merk','Soort','Type','Kleur','Standaardprijs','Inkoopprijs']]

    fv = read_table(con, 'Fietsverkoop_Fiets')
    fv['BronSysteem'] = 'Fiets_Verkoop'; fv['ProductType'] = 'Fiets'; fv['Naam'] = None
    fv = fv.rename(columns={'fietsnr':'ProductNr','type':'Type','standaardprijs':'Standaardprijs','inkoopprijs':'Inkoopprijs','merk':'Merk','soort':'Soort','kleur':'Kleur'})
    fv = fv[['BronSysteem','ProductNr','ProductType','Naam','Merk','Soort','Type','Kleur','Standaardprijs','Inkoopprijs']]

    of = read_table(con, 'Onderhoud_Fiets')
    of['BronSysteem'] = 'Onderhoud'; of['ProductType'] = 'Fiets'; of['Naam'] = None
    of = of.rename(columns={'fietsnr':'ProductNr','type':'Type','standaardprijs':'Standaardprijs','inkoopprijs':'Inkoopprijs','merk':'Merk','soort':'Soort','kleur':'Kleur'})
    of = of[['BronSysteem','ProductNr','ProductType','Naam','Merk','Soort','Type','Kleur','Standaardprijs','Inkoopprijs']]

    ai = read_table(con, 'Accessoire_Inkoop_Accessoire')
    ai['BronSysteem'] = 'Accessoire_Inkoop'; ai['ProductType'] = 'Accessoire'; ai['Merk'] = None; ai['Type'] = None; ai['Kleur'] = None
    ai = ai.rename(columns={'accessoirenr':'ProductNr','naam':'Naam','standaardprijs':'Standaardprijs','inkoopprijs':'Inkoopprijs','soort':'Soort'})
    ai = ai[['BronSysteem','ProductNr','ProductType','Naam','Merk','Soort','Type','Kleur','Standaardprijs','Inkoopprijs']]

    av = read_table(con, 'Accessoireverkoop_Accessoire')
    av['BronSysteem'] = 'Accessoire_Verkoop'; av['ProductType'] = 'Accessoire'; av['Merk'] = None; av['Type'] = None; av['Kleur'] = None
    av = av.rename(columns={'accessoirenr':'ProductNr','naam':'Naam','standaardprijs':'Standaardprijs','inkoopprijs':'Inkoopprijs','soort':'Soort'})
    av = av[['BronSysteem','ProductNr','ProductType','Naam','Merk','Soort','Type','Kleur','Standaardprijs','Inkoopprijs']]

    dim_product = pd.concat([fi, fv, of, ai, av], ignore_index=True).drop_duplicates()

    fif = read_table(con, 'Fiets_Inkoop_Fabrikant'); fif['BronSysteem'] = 'Fiets_Inkoop'; fif['PartnerType'] = 'Fabrikant'
    fif = fif.rename(columns={'fabrikantnr':'PartnerNr','naam':'Naam','adres':'Adres','plaats':'Plaats'})
    fif = fif[['BronSysteem','PartnerNr','PartnerType','Naam','Adres','Plaats']]

    ail = read_table(con, 'Accessoire_Inkoop_Leverancier'); ail['BronSysteem'] = 'Accessoire_Inkoop'; ail['PartnerType'] = 'Leverancier'
    ail = ail.rename(columns={'leveranciernr':'PartnerNr','naam':'Naam','adres':'Adres','woonplaats':'Plaats'})
    ail = ail[['BronSysteem','PartnerNr','PartnerType','Naam','Adres','Plaats']]

    dim_partner = pd.concat([fif, ail], ignore_index=True).drop_duplicates()

    kv = read_table(con, 'Fietsverkoop_Klant'); kv['BronSysteem'] = 'Fiets_Verkoop'
    kv = kv.rename(columns={'klantnr':'KlantNr','naam':'Naam','adres':'Adres','woonplaats':'Woonplaats','geslacht':'Geslacht','geboortedatum':'Geboortedatum'})

    ka = read_table(con, 'Accessoireverkoop_Klant'); ka['BronSysteem'] = 'Accessoire_Verkoop'
    ka = ka.rename(columns={'klantnr':'KlantNr','naam':'Naam','adres':'Adres','woonplaats':'Woonplaats','geslacht':'Geslacht','geboortedatum':'Geboortedatum'})

    dim_klant = pd.concat([kv, ka], ignore_index=True)
    dim_klant['Leeftijd'] = dim_klant['Geboortedatum'].apply(age_from_birthdate)
    dim_klant['Leeftijdscategorie'] = dim_klant['Leeftijd'].apply(age_cat)
    dim_klant = dim_klant[['BronSysteem','KlantNr','Naam','Adres','Woonplaats','Geslacht','Geboortedatum','Leeftijd','Leeftijdscategorie']].drop_duplicates()

    mv = read_table(con, 'Fietsverkoop_Monteur'); mv['BronSysteem'] = 'Fiets_Verkoop'
    mv = mv.rename(columns={'monteurnr':'MonteurNr','naam':'Naam','woonplaats':'Woonplaats','uurloon':'Uurloon','filiaal':'FiliaalNr'})

    ma = read_table(con, 'Accessoireverkoop_Monteur'); ma['BronSysteem'] = 'Accessoire_Verkoop'
    ma = ma.rename(columns={'monteurnr':'MonteurNr','naam':'Naam','woonplaats':'Woonplaats','uurloon':'Uurloon','filiaal':'FiliaalNr'})

    mo = read_table(con, 'Onderhoud_Monteur'); mo['BronSysteem'] = 'Onderhoud'
    mo = mo.rename(columns={'monteurnr':'MonteurNr','naam':'Naam','woonplaats':'Woonplaats','uurloon':'Uurloon','filiaal':'FiliaalNr'})

    dim_monteur = pd.concat([mv, ma, mo], ignore_index=True)
    dim_monteur = dim_monteur[['BronSysteem','MonteurNr','Naam','Woonplaats','Uurloon','FiliaalNr']].drop_duplicates()

    fvf = read_table(con, 'Fietsverkoop_Filiaal'); fvf['BronSysteem'] = 'Fiets_Verkoop'
    fvf = fvf.rename(columns={'filiaalnr':'FiliaalNr','naam':'Naam','adres':'Adres','provincie':'Provincie'})

    avf = read_table(con, 'Accessoireverkoop_Filiaal'); avf['BronSysteem'] = 'Accessoire_Verkoop'
    avf = avf.rename(columns={'filiaalnr':'FiliaalNr','naam':'Naam','adres':'Adres','provincie':'Provincie'})

    ofi = read_table(con, 'Onderhoud_Filiaal'); ofi['BronSysteem'] = 'Onderhoud'
    ofi = ofi.rename(columns={'filiaalnr':'FiliaalNr','naam':'Naam','adres':'Adres','provincie':'Provincie'})

    dim_filiaal = pd.concat([fvf, avf, ofi], ignore_index=True)
    dim_filiaal = dim_filiaal[['BronSysteem','FiliaalNr','Naam','Adres','Provincie']].drop_duplicates()

    dates = []
    for q in [
        "SELECT datum AS Datum FROM Fietsverkoop_Fiets_Verkoop",
        "SELECT datum AS Datum FROM Accessoireverkoop_Accessoire_Verkoop",
        "SELECT substr(datum,1,10) AS Datum FROM Onderhoud"
    ]:
        dates.append(pd.read_sql_query(q, con))

    inkoop_fi = read_table(con, 'Fiets_Inkoop')[['inkoopmaand','inkoopjaar']].drop_duplicates()
    inkoop_fi['Datum'] = inkoop_fi.apply(lambda r: f"{int(r['inkoopjaar']):04d}-{int(r['inkoopmaand']):02d}-01", axis=1)
    dates.append(inkoop_fi[['Datum']])

    inkoop_ai = read_table(con, 'Accessoire_Inkoop')[['inkoopmaand','inkoopjaar']].drop_duplicates()
    inkoop_ai['Datum'] = inkoop_ai.apply(lambda r: f"{int(r['inkoopjaar']):04d}-{int(r['inkoopmaand']):02d}-01", axis=1)
    dates.append(inkoop_ai[['Datum']])

    dim_datum = pd.concat(dates, ignore_index=True).dropna().drop_duplicates()
    dt_series = pd.to_datetime(dim_datum['Datum'])
    dim_datum['Dag'] = dt_series.dt.day
    dim_datum['Maand'] = dt_series.dt.month
    dim_datum['Jaar'] = dt_series.dt.year
    dim_datum['Kwartaal'] = dt_series.dt.quarter
    dim_datum['Weekdag'] = dt_series.dt.day_name()
    dim_datum['Seizoen'] = dim_datum['Maand'].apply(season)
    dim_datum['IsWeekend'] = dt_series.dt.dayofweek.isin([5,6]).astype(int)
    dim_datum = dim_datum[['Datum','Dag','Maand','Jaar','Kwartaal','Weekdag','Seizoen','IsWeekend']].sort_values('Datum').reset_index(drop=True)

    ot = read_table(con, 'Onderhoud')[['starttijd','eindtijd']]
    times = pd.DataFrame({'Tijd': pd.concat([ot['starttijd'], ot['eindtijd']], ignore_index=True).drop_duplicates().map(normalize_time)})
    tm = pd.to_datetime(times['Tijd'], format='%H:%M:%S')
    times['Uur'] = tm.dt.hour
    times['Minuut'] = tm.dt.minute
    times['Dagdeel'] = times['Uur'].apply(dagdeel)
    dim_tijd = times[['Tijd','Uur','Minuut','Dagdeel']].sort_values(['Uur','Minuut']).reset_index(drop=True)

    fink = read_table(con, 'Fiets_Inkoop').merge(read_table(con, 'Fiets_Inkoop_Fiets'), left_on='fiets', right_on='fietsnr')
    fact_inkoop_fiets = pd.DataFrame({
        'BronFeit':'Fiets_Inkoop',
        'InkoopNr':fink['inkoopnr'],
        'ProductBronSysteem':'Fiets_Inkoop',
        'ProductType':'Fiets',
        'ProductNr':fink['fiets'],
        'PartnerBronSysteem':'Fiets_Inkoop',
        'PartnerType':'Fabrikant',
        'PartnerNr':fink['fabrikant'],
        'Datum':fink.apply(lambda r: f"{int(r['inkoopjaar']):04d}-{int(r['inkoopmaand']):02d}-01", axis=1),
        'Aantal':fink['aantal'],
        'Inkoopprijs':fink['inkoopprijs'],
        'Inkoopbedrag':(fink['aantal'] * fink['inkoopprijs']).round(2),
        'KortingBedrag':((fink['standaardprijs'] - fink['inkoopprijs']) * fink['aantal']).round(2)
    })

    aink = read_table(con, 'Accessoire_Inkoop').merge(read_table(con, 'Accessoire_Inkoop_Accessoire'), left_on='accessoire', right_on='accessoirenr')
    fact_inkoop_acc = pd.DataFrame({
        'BronFeit':'Accessoire_Inkoop',
        'InkoopNr':aink['inkoopnr'],
        'ProductBronSysteem':'Accessoire_Inkoop',
        'ProductType':'Accessoire',
        'ProductNr':aink['accessoire'],
        'PartnerBronSysteem':'Accessoire_Inkoop',
        'PartnerType':'Leverancier',
        'PartnerNr':aink['leverancier'],
        'Datum':aink.apply(lambda r: f"{int(r['inkoopjaar']):04d}-{int(r['inkoopmaand']):02d}-01", axis=1),
        'Aantal':aink['aantal'],
        'Inkoopprijs':aink['inkoopprijs'],
        'Inkoopbedrag':(aink['aantal'] * aink['inkoopprijs']).round(2),
        'KortingBedrag':((aink['standaardprijs'] - aink['inkoopprijs']) * aink['aantal']).round(2)
    })
    fact_inkoop = pd.concat([fact_inkoop_fiets, fact_inkoop_acc], ignore_index=True)

    bikev = read_table(con, 'Fietsverkoop_Fiets_Verkoop') \
        .merge(read_table(con, 'Fietsverkoop_Fiets'), left_on='fiets', right_on='fietsnr') \
        .merge(read_table(con, 'Fietsverkoop_Monteur')[['monteurnr','filiaal']], left_on='monteur', right_on='monteurnr')
    fact_verkoop_bike = pd.DataFrame({
        'BronFeit':'Fiets_Verkoop',
        'VerkoopNr':bikev['fiets_verkoopnr'],
        'ProductBronSysteem':'Fiets_Verkoop',
        'ProductType':'Fiets',
        'ProductNr':bikev['fiets'],
        'KlantBronSysteem':'Fiets_Verkoop',
        'KlantNr':bikev['klant'],
        'MonteurBronSysteem':'Fiets_Verkoop',
        'MonteurNr':bikev['monteur'],
        'FiliaalBronSysteem':'Fiets_Verkoop',
        'FiliaalNr':bikev['filiaal'],
        'Datum':bikev['datum'],
        'Aantal':bikev['aantal'],
        'Verkoopprijs':bikev['verkoopprijs'],
        'Omzet':(bikev['aantal'] * bikev['verkoopprijs']).round(2),
        'Inkoopbedrag':(bikev['aantal'] * bikev['inkoopprijs']).round(2),
        'Brutowinst':((bikev['aantal'] * bikev['verkoopprijs']) - (bikev['aantal'] * bikev['inkoopprijs'])).round(2)
    })

    accv = read_table(con, 'Accessoireverkoop_Accessoire_Verkoop') \
        .merge(read_table(con, 'Accessoireverkoop_Accessoire'), left_on='accessoire', right_on='accessoirenr') \
        .merge(read_table(con, 'Accessoireverkoop_Monteur')[['monteurnr','filiaal']], left_on='monteur', right_on='monteurnr')
    fact_verkoop_acc = pd.DataFrame({
        'BronFeit':'Accessoire_Verkoop',
        'VerkoopNr':accv['accessoire_verkoopnr'],
        'ProductBronSysteem':'Accessoire_Verkoop',
        'ProductType':'Accessoire',
        'ProductNr':accv['accessoire'],
        'KlantBronSysteem':'Accessoire_Verkoop',
        'KlantNr':accv['klant'],
        'MonteurBronSysteem':'Accessoire_Verkoop',
        'MonteurNr':accv['monteur'],
        'FiliaalBronSysteem':'Accessoire_Verkoop',
        'FiliaalNr':accv['filiaal'],
        'Datum':accv['datum'],
        'Aantal':accv['aantal'],
        'Verkoopprijs':accv['verkoopprijs'],
        'Omzet':(accv['aantal'] * accv['verkoopprijs']).round(2),
        'Inkoopbedrag':(accv['aantal'] * accv['inkoopprijs']).round(2),
        'Brutowinst':((accv['aantal'] * accv['verkoopprijs']) - (accv['aantal'] * accv['inkoopprijs'])).round(2)
    })
    fact_verkoop = pd.concat([fact_verkoop_bike, fact_verkoop_acc], ignore_index=True)

    ond = read_table(con, 'Onderhoud').merge(read_table(con, 'Onderhoud_Monteur')[['monteurnr','filiaal','uurloon']], left_on='monteur', right_on='monteurnr')
    fact_onderhoud = pd.DataFrame({
        'BronFeit':'Onderhoud',
        'OnderhoudNr':ond['onderhoudnr'],
        'ProductBronSysteem':'Onderhoud',
        'ProductType':'Fiets',
        'ProductNr':ond['fiets'],
        'MonteurBronSysteem':'Onderhoud',
        'MonteurNr':ond['monteur'],
        'FiliaalBronSysteem':'Onderhoud',
        'FiliaalNr':ond['filiaal'],
        'Datum':ond['datum'].astype(str).str[:10],
        'StartTijd':ond['starttijd'].map(normalize_time),
        'EindTijd':ond['eindtijd'].map(normalize_time),
        'AantalOnderhoud':1,
        'OnderhoudsduurMin':ond.apply(lambda r: minutes_between(r['starttijd'], r['eindtijd']), axis=1),
        'Arbeidskosten':ond.apply(lambda r: round((minutes_between(r['starttijd'], r['eindtijd']) / 60.0) * float(r['uurloon']), 2), axis=1)
    })

    con.close()
    return {
        'Dim_Product': dim_product,
        'Dim_Partner': dim_partner,
        'Dim_Klant': dim_klant,
        'Dim_Monteur': dim_monteur,
        'Dim_Filiaal': dim_filiaal,
        'Dim_Datum': dim_datum,
        'Dim_Tijd': dim_tijd,
        'Fact_Inkoop': fact_inkoop,
        'Fact_Verkoop': fact_verkoop,
        'Fact_Onderhoud': fact_onderhoud
    }

src = extract_sources(SDM_DB)
{k: v.shape for k, v in src.items()}

DatabaseError: Execution failed on sql 'SELECT * FROM Fiets_Inkoop_Fiets': no such table: Fiets_Inkoop_Fiets

In [ ]:
# Delta-detectie en SCD-logica
def norm_value(v):
    if pd.isna(v):
        return None
    if isinstance(v, float):
        if math.isnan(v):
            return None
        return round(v, 8)
    return v

def detect_deltas(source_df, target_df, business_keys, compare_cols):
    if target_df.empty:
        empty = source_df.iloc[0:0].copy()
        return {'new': source_df.reset_index(drop=True), 'changed': empty, 'unchanged': empty}

    merged = source_df.merge(target_df, on=business_keys, how='left', suffixes=('_src', '_tgt'), indicator=True)
    new_mask = merged['_merge'] == 'left_only'
    both_mask = merged['_merge'] == 'both'

    changed = np.array([False] * len(merged))
    for col in compare_cols:
        src_col = f'{col}_src' if f'{col}_src' in merged.columns else col
        tgt_col = f'{col}_tgt' if f'{col}_tgt' in merged.columns else col
        diffs = [norm_value(a) != norm_value(b) for a, b in zip(merged[src_col], merged[tgt_col])]
        changed = changed | np.array(diffs)

    changed_mask = both_mask & changed
    unchanged_mask = both_mask & (~changed)

    cols = source_df.columns.tolist()
    def extract(mask):
        selected = []
        for c in cols:
            if c in business_keys:
                selected.append(c)
            elif f'{c}_src' in merged.columns:
                selected.append(f'{c}_src')
            else:
                selected.append(c)
        out = merged.loc[mask, selected].copy()
        out.columns = cols
        return out.reset_index(drop=True)

    return {'new': extract(new_mask), 'changed': extract(changed_mask), 'unchanged': extract(unchanged_mask)}

def upsert_dim_scd1(con, table, source_df, business_keys, compare_cols):
    target_df = pd.read_sql_query(f"SELECT * FROM {table}", con)
    target_df = target_df[business_keys + compare_cols] if not target_df.empty else pd.DataFrame(columns=business_keys + compare_cols)
    deltas = detect_deltas(source_df, target_df, business_keys, compare_cols)

    cols = source_df.columns.tolist()
    cur = con.cursor()
    for _, row in deltas['new'].iterrows():
        cur.execute(
            f"INSERT INTO {table} ({', '.join(cols)}) VALUES ({', '.join(['?'] * len(cols))})",
            [None if pd.isna(row[c]) else row[c] for c in cols]
        )
    for _, row in deltas['changed'].iterrows():
        cur.execute(
            f"UPDATE {table} SET {', '.join([f'{c}=?' for c in compare_cols])} WHERE {' AND '.join([f'{k}=?' for k in business_keys])}",
            [None if pd.isna(row[c]) else row[c] for c in compare_cols] + [row[k] for k in business_keys]
        )
    con.commit()
    return {k: len(v) for k, v in deltas.items()}

def upsert_dim_scd2(con, table, source_df, business_keys, compare_cols, load_ts):
    target_df = pd.read_sql_query(f"SELECT * FROM {table} WHERE IsCurrent = 1", con)
    target_df = target_df[business_keys + compare_cols] if not target_df.empty else pd.DataFrame(columns=business_keys + compare_cols)
    deltas = detect_deltas(source_df, target_df, business_keys, compare_cols)

    cols = source_df.columns.tolist()
    insert_cols = cols + ['ValidFrom', 'ValidTo', 'IsCurrent']
    cur = con.cursor()

    for _, row in deltas['new'].iterrows():
        cur.execute(
            f"INSERT INTO {table} ({', '.join(insert_cols)}) VALUES ({', '.join(['?'] * len(insert_cols))})",
            [None if pd.isna(row[c]) else row[c] for c in cols] + [load_ts, None, 1]
        )

    for _, row in deltas['changed'].iterrows():
        cur.execute(
            f"UPDATE {table} SET ValidTo = ?, IsCurrent = 0 WHERE {' AND '.join([f'{k}=?' for k in business_keys])} AND IsCurrent = 1",
            [load_ts] + [row[k] for k in business_keys]
        )
        cur.execute(
            f"INSERT INTO {table} ({', '.join(insert_cols)}) VALUES ({', '.join(['?'] * len(insert_cols))})",
            [None if pd.isna(row[c]) else row[c] for c in cols] + [load_ts, None, 1]
        )

    con.commit()
    return {k: len(v) for k, v in deltas.items()}

def get_mapping(con, table, key_cols, surrogate_key, current_only=False):
    q = f"SELECT {', '.join(key_cols + [surrogate_key])} FROM {table}"
    if current_only:
        q += " WHERE IsCurrent = 1"
    df = pd.read_sql_query(q, con)
    mapping = {}
    for _, row in df.iterrows():
        key = row[key_cols[0]] if len(key_cols) == 1 else tuple(row[c] for c in key_cols)
        mapping[key] = row[surrogate_key]
    return mapping

def insert_new_facts(con, table, source_df, business_keys):
    target_df = pd.read_sql_query(f"SELECT * FROM {table}", con)
    target_df = target_df[business_keys] if not target_df.empty else pd.DataFrame(columns=business_keys)
    deltas = detect_deltas(source_df, target_df, business_keys, [])

    cols = source_df.columns.tolist()
    cur = con.cursor()
    for _, row in deltas['new'].iterrows():
        cur.execute(
            f"INSERT INTO {table} ({', '.join(cols)}) VALUES ({', '.join(['?'] * len(cols))})",
            [None if pd.isna(row[c]) else row[c] for c in cols]
        )
    con.commit()
    return {k: len(v) for k, v in deltas.items()}

In [ ]:
# ETL uitvoeren
prepare_target_schema(TARGET_DB)
src = extract_sources(SDM_DB)

con = connect(TARGET_DB)
etl_result = {}

# SCD Type 1
etl_result['Dim_Datum'] = upsert_dim_scd1(con, 'Dim_Datum', src['Dim_Datum'], ['Datum'], ['Dag','Maand','Jaar','Kwartaal','Weekdag','Seizoen','IsWeekend'])
etl_result['Dim_Tijd'] = upsert_dim_scd1(con, 'Dim_Tijd', src['Dim_Tijd'], ['Tijd'], ['Uur','Minuut','Dagdeel'])
etl_result['Dim_Partner'] = upsert_dim_scd1(con, 'Dim_Partner', src['Dim_Partner'], ['BronSysteem','PartnerType','PartnerNr'], ['Naam','Adres','Plaats'])
etl_result['Dim_Filiaal'] = upsert_dim_scd1(con, 'Dim_Filiaal', src['Dim_Filiaal'], ['BronSysteem','FiliaalNr'], ['Naam','Adres','Provincie'])
etl_result['Dim_Monteur'] = upsert_dim_scd1(con, 'Dim_Monteur', src['Dim_Monteur'][['BronSysteem','MonteurNr','Naam','Woonplaats','Uurloon']], ['BronSysteem','MonteurNr'], ['Naam','Woonplaats','Uurloon'])

# SCD Type 2
etl_result['Dim_Product'] = upsert_dim_scd2(con, 'Dim_Product', src['Dim_Product'], ['BronSysteem','ProductType','ProductNr'], ['Naam','Merk','Soort','Type','Kleur','Standaardprijs','Inkoopprijs'], LOAD_TS)
etl_result['Dim_Klant'] = upsert_dim_scd2(con, 'Dim_Klant', src['Dim_Klant'], ['BronSysteem','KlantNr'], ['Naam','Adres','Woonplaats','Geslacht','Geboortedatum','Leeftijd','Leeftijdscategorie'], LOAD_TS)

# Surrogate key lookups
datum_map = get_mapping(con, 'Dim_Datum', ['Datum'], 'DatumKey')
tijd_map = get_mapping(con, 'Dim_Tijd', ['Tijd'], 'TijdKey')
partner_map = get_mapping(con, 'Dim_Partner', ['BronSysteem','PartnerType','PartnerNr'], 'PartnerKey')
filiaal_map = get_mapping(con, 'Dim_Filiaal', ['BronSysteem','FiliaalNr'], 'FiliaalKey')
monteur_map = get_mapping(con, 'Dim_Monteur', ['BronSysteem','MonteurNr'], 'MonteurKey')
product_map = get_mapping(con, 'Dim_Product', ['BronSysteem','ProductType','ProductNr'], 'ProductKey', current_only=True)
klant_map = get_mapping(con, 'Dim_Klant', ['BronSysteem','KlantNr'], 'KlantKey', current_only=True)

# Fact_Inkoop
f = src['Fact_Inkoop'].copy()
f['ProductKey'] = f.apply(lambda r: product_map[(r['ProductBronSysteem'], r['ProductType'], r['ProductNr'])], axis=1)
f['PartnerKey'] = f.apply(lambda r: partner_map[(r['PartnerBronSysteem'], r['PartnerType'], r['PartnerNr'])], axis=1)
f['DatumKey'] = f['Datum'].map(datum_map)
fact_inkoop = f[['BronFeit','InkoopNr','ProductKey','PartnerKey','DatumKey','Aantal','Inkoopprijs','Inkoopbedrag','KortingBedrag']]
etl_result['Fact_Inkoop'] = insert_new_facts(con, 'Fact_Inkoop', fact_inkoop, ['BronFeit','InkoopNr'])

# Fact_Verkoop
f = src['Fact_Verkoop'].copy()
f['ProductKey'] = f.apply(lambda r: product_map[(r['ProductBronSysteem'], r['ProductType'], r['ProductNr'])], axis=1)
f['KlantKey'] = f.apply(lambda r: klant_map[(r['KlantBronSysteem'], r['KlantNr'])], axis=1)
f['MonteurKey'] = f.apply(lambda r: monteur_map[(r['MonteurBronSysteem'], r['MonteurNr'])], axis=1)
f['FiliaalKey'] = f.apply(lambda r: filiaal_map[(r['FiliaalBronSysteem'], r['FiliaalNr'])], axis=1)
f['DatumKey'] = f['Datum'].map(datum_map)
fact_verkoop = f[['BronFeit','VerkoopNr','ProductKey','KlantKey','MonteurKey','FiliaalKey','DatumKey','Aantal','Verkoopprijs','Omzet','Inkoopbedrag','Brutowinst']]
etl_result['Fact_Verkoop'] = insert_new_facts(con, 'Fact_Verkoop', fact_verkoop, ['BronFeit','VerkoopNr'])

# Fact_Onderhoud
f = src['Fact_Onderhoud'].copy()
f['ProductKey'] = f.apply(lambda r: product_map[(r['ProductBronSysteem'], r['ProductType'], r['ProductNr'])], axis=1)
f['MonteurKey'] = f.apply(lambda r: monteur_map[(r['MonteurBronSysteem'], r['MonteurNr'])], axis=1)
f['FiliaalKey'] = f.apply(lambda r: filiaal_map[(r['FiliaalBronSysteem'], r['FiliaalNr'])], axis=1)
f['DatumKey'] = f['Datum'].map(datum_map)
f['StartTijdKey'] = f['StartTijd'].map(tijd_map)
f['EindTijdKey'] = f['EindTijd'].map(tijd_map)
fact_onderhoud = f[['BronFeit','OnderhoudNr','ProductKey','MonteurKey','FiliaalKey','DatumKey','StartTijdKey','EindTijdKey','AantalOnderhoud','OnderhoudsduurMin','Arbeidskosten']]
etl_result['Fact_Onderhoud'] = insert_new_facts(con, 'Fact_Onderhoud', fact_onderhoud, ['OnderhoudNr'])

print(etl_result)

In [ ]:
# Controleren van het geladen resultaat
table_names = [
    'Dim_Datum', 'Dim_Tijd', 'Dim_Partner', 'Dim_Filiaal',
    'Dim_Monteur', 'Dim_Product', 'Dim_Klant',
    'Fact_Inkoop', 'Fact_Verkoop', 'Fact_Onderhoud'
]

counts = []
for table_name in table_names:
    row_count = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    counts.append({'Tabel': table_name, 'Aantal rijen': row_count})

counts_df = pd.DataFrame(counts)
counts_df

In [ ]:
# Tweede run ter controle
# Herhaal exact dezelfde load om te bewijzen dat de delta-detectie werkt.
rerun_result = {}
rerun_result['Dim_Datum'] = upsert_dim_scd1(con, 'Dim_Datum', src['Dim_Datum'], ['Datum'], ['Dag','Maand','Jaar','Kwartaal','Weekdag','Seizoen','IsWeekend'])
rerun_result['Dim_Tijd'] = upsert_dim_scd1(con, 'Dim_Tijd', src['Dim_Tijd'], ['Tijd'], ['Uur','Minuut','Dagdeel'])
rerun_result['Dim_Partner'] = upsert_dim_scd1(con, 'Dim_Partner', src['Dim_Partner'], ['BronSysteem','PartnerType','PartnerNr'], ['Naam','Adres','Plaats'])
rerun_result['Dim_Filiaal'] = upsert_dim_scd1(con, 'Dim_Filiaal', src['Dim_Filiaal'], ['BronSysteem','FiliaalNr'], ['Naam','Adres','Provincie'])
rerun_result['Dim_Monteur'] = upsert_dim_scd1(con, 'Dim_Monteur', src['Dim_Monteur'][['BronSysteem','MonteurNr','Naam','Woonplaats','Uurloon']], ['BronSysteem','MonteurNr'], ['Naam','Woonplaats','Uurloon'])
rerun_result['Dim_Product'] = upsert_dim_scd2(con, 'Dim_Product', src['Dim_Product'], ['BronSysteem','ProductType','ProductNr'], ['Naam','Merk','Soort','Type','Kleur','Standaardprijs','Inkoopprijs'], LOAD_TS)
rerun_result['Dim_Klant'] = upsert_dim_scd2(con, 'Dim_Klant', src['Dim_Klant'], ['BronSysteem','KlantNr'], ['Naam','Adres','Woonplaats','Geslacht','Geboortedatum','Leeftijd','Leeftijdscategorie'], LOAD_TS)
rerun_result['Fact_Inkoop'] = insert_new_facts(con, 'Fact_Inkoop', fact_inkoop, ['BronFeit','InkoopNr'])
rerun_result['Fact_Verkoop'] = insert_new_facts(con, 'Fact_Verkoop', fact_verkoop, ['BronFeit','VerkoopNr'])
rerun_result['Fact_Onderhoud'] = insert_new_facts(con, 'Fact_Onderhoud', fact_onderhoud, ['OnderhoudNr'])

rerun_result

In [ ]:
con.close()
print(f'ETL klaar. Resultaat staat in: {TARGET_DB}')